# Sino-Nom OCR — Kaggle GPU

**Cấu hình notebook bắt buộc:**

| Setting | Giá trị |
|---|---|
| Accelerator | **GPU T4 x2** (hoặc P100) |
| Internet | **ON** — cần để pip install và tải model PaddleOCR |

Đã kiểm trên Kaggle: Python 3.12, paddle 3.3.1 (wheel cp312), numpy 2.x.

Chỉ cài 7 package mà luồng OCR thực sự cần, **không** cài cả `requirements.txt`
(`underthesea`, `google-generativeai`, `scikit-learn`... chỉ dùng cho bước
segment/align, cài thêm vừa lâu vừa dễ hỏng trên 3.12).

## 1. Clone source

In [ ]:
!rm -rf /kaggle/working/SinoNom-NLP
!git clone -q https://github.com/quachthanhhmd/SinoNom-NLP /kaggle/working/SinoNom-NLP
%cd /kaggle/working/SinoNom-NLP
!git checkout -q feat/paddle-ocr-enhance && git pull -q

# Xác nhận đang ở đúng commit có bản sửa
!git log --oneline -1
print()
import os
for f in ['main.py', 'ocr/ocr_pipeline.py', 'run_config.json']:
    print(('  OK  ' if os.path.exists(f) else '  THIẾU  ') + f)

In [ ]:
# Bản sửa mới có mặt chưa? Nếu FAIL -> code chưa được push, đừng chạy tiếp.
src = open('ocr/ocr_pipeline.py', encoding='utf-8').read()
checks = {
    'build_ocr_engine (tắt doc_unwarping)': 'use_doc_unwarping=False' in src,
    'OpenCC giản->phồn'                   : 'def to_traditional' in src,
    'retry nhị phân khi ảo giác'          : 'def _looks_hallucinated' in src,
    'chẻ ô lưới rộng'                     : 'def _subdivide_wide_cells' in src,
    'cắt viền khung'                      : 'def find_frame_rows' in src,
    'giữ bản tâm ở mép'                   : '_BANXIN_HINT' in src,
    'PDF render 300 dpi'                  : 'dpi=300' in open('main.py', encoding='utf-8').read(),
}
for k, v in checks.items():
    print(f"  {'OK  ' if v else 'FAIL'}  {k}")
assert all(checks.values()), 'Source cũ! Hãy commit + push bản sửa rồi chạy lại cell clone.'

## 2. Cài dependency (GPU)

`paddlepaddle-gpu` trên PyPI **không kèm CUDA** — phải cài từ index riêng của
Paddle, đúng phiên bản CUDA của máy. Cell dưới tự dò qua `nvcc`.

In [ ]:
import re, subprocess, sys

print('Python', sys.version.split()[0])
nvcc = subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout
m = re.search(r'release (\d+)\.(\d+)', nvcc)
major, minor = (int(m.group(1)), int(m.group(2))) if m else (12, 6)
tag = ('cu130' if major >= 13 else
       'cu129' if (major, minor) >= (12, 9) else
       'cu126' if (major, minor) >= (12, 6) else 'cu118')
print(f'CUDA {major}.{minor} -> index {tag}')

In [ ]:
# Gỡ bản CPU (Kaggle có thể cài sẵn) rồi cài bản GPU
!pip uninstall -y -q paddlepaddle paddlepaddle-gpu
!pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/{tag}/

# 6 package còn lại mà luồng OCR cần
!pip install -q "paddleocr>=3.7.0" scipy opencc-python-reimplemented pymupdf python-dotenv "opencv-python-headless"

## 3. Kiểm tra GPU — **đừng bỏ qua**

Nếu Paddle không thấy GPU, nó vẫn chạy bằng CPU, **không báo lỗi**, chỉ chậm
hơn khoảng 10 lần. `assert` ở đây để anh biết ngay thay vì phát hiện sau 1 tiếng.

In [ ]:
import sys, importlib
print('Python  ', sys.version.split()[0])

import paddle, paddleocr, numpy, scipy, cv2, fitz, opencc
print('paddle  ', paddle.__version__)
print('paddleocr', paddleocr.__version__)
print('numpy   ', numpy.__version__)
print('scipy   ', scipy.__version__)
print('opencv  ', cv2.__version__)
print('pymupdf ', fitz.__doc__ and __import__('pymupdf').__version__)
print('opencc  ', opencc.OpenCC('s2t').convert('务国广') , '(phải ra 務國廣)')

cuda = paddle.device.is_compiled_with_cuda()
n_gpu = paddle.device.cuda.device_count() if cuda else 0
print(f'\ncompiled_with_cuda={cuda}  gpu_count={n_gpu}')
assert cuda and n_gpu > 0, 'Paddle KHÔNG thấy GPU — kiểm tra Accelerator và cell cài đặt!'

# Pipeline import được không (scipy hay thiếu -> ocr_pipeline chết ngay từ import)
sys.path.insert(0, '/kaggle/working/SinoNom-NLP')
from ocr.ocr_pipeline import ocr_sinonom_page, build_ocr_engine
print('import ocr_pipeline OK')

## 4. Tải dataset ảnh từ Google Drive

Bỏ qua cell này nếu chỉ chạy OCR cho PDF (PDF đã nằm sẵn trong repo).

In [ ]:
FILE_ID = '1VHR61FGXMUANgmaLEnPYYgU5ygiY3Z72'   # file .zip công khai trên Drive

!pip install -q gdown
!gdown -q --id $FILE_ID
!ls -F *.zip
!unzip -q -o china.zip -d china
!ls china

## 5. Chạy thử 5 trang trước

Xem log rồi mới chạy full. Mỗi trang nên ra khoảng 20–30 cột và 400–480 chữ
(ảnh), hoặc 8–15 cột (PDF nửa tờ). Nếu số cột chỉ còn một chữ số, hoặc chữ
toàn 務/局/司 thì **dừng lại**, đừng chạy full.

In [ ]:
# --- Thử ẢNH: 5 trang của q1 ---
!python scripts/run_han_ocr.py --input ./china/china/q1 --limit 5 --out /kaggle/working/smoke_q1.json

In [ ]:
# --- Thử PDF: 5 trang của 14_15.pdf ---
!python main.py --do-ocr --han_pdf_dir ./dataset/china/han_pdf --ocr-pdf 14_15.pdf --first-n-images 5

## 6. Chạy full

In [ ]:
# --- ẢNH, chia đôi cho 2 GPU chạy song song ---
import subprocess, time, paddle

VOLUME = 'q1'
IN_DIR = f'./china/china/{VOLUME}'
OUT    = f'/kaggle/working/{VOLUME}.json'
n      = paddle.device.cuda.device_count()

def run(cmd):
    print('$', cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode

t0 = time.time()
if n >= 2:
    ps = [subprocess.Popen(
              f'python scripts/run_han_ocr.py --input {IN_DIR} --out {OUT} '
              f'--device gpu:{i} --shard {i}/{n}', shell=True)
          for i in range(n)]
    codes = [p.wait() for p in ps]
    print('exit codes:', codes)
    if any(codes):
        print('!! có shard lỗi — xem log phía trên trước khi dùng kết quả')
    run(f'python scripts/merge_shards.py --out {OUT}')
else:
    run(f'python scripts/run_han_ocr.py --input {IN_DIR} --out {OUT} --device gpu:0')

print(f'\nTổng {time.time()-t0:.0f}s')

In [ ]:
# --- PDF (main.py chỉ dùng 1 GPU, chưa hỗ trợ chia shard) ---
# Bỏ --ocr-pdf để chạy hết mọi PDF trong thư mục.
!python main.py --do-ocr --han_pdf_dir ./dataset/china/han_pdf --ocr-pdf 14_15.pdf
!ls -la output/han_ocr/

## 7. Kiểm tra chất lượng

In [ ]:
import json, re, glob, collections

CJK   = re.compile(r'[\u4e00-\u9fff\u3400-\u4dbf]')
NOISE = set('務局司商創財員市房品號機業科貿發銀店濟')   # từ vựng hiện đại model hay bịa
SIMP  = set('务济国广见灵员业动华长间')                  # chữ giản thể còn sót
rep   = lambda t: 1 - len(set(t)) / len(t) if t else 0

def bad(t):
    c = CJK.findall(t)
    return len(c) >= 4 and (sum(x in NOISE for x in c) / len(c) >= .30 or rep(t) >= .45)

KEYS = ('bbox', 'volume', 'page', 'page_number', 'text', 'middle', 'consider')

def report(path):
    d = json.load(open(path, encoding='utf-8'))
    if not d:
        print(f'{path}: RỖNG'); return
    n_bad  = sum(bad(r['text']) for r in d)
    n_simp = sum(1 for r in d for ch in r['text'] if ch in SIMP)
    print(f'\n{path}')
    print(f'  {len({r["page"] for r in d})} trang, {len(d)} cột, {sum(len(r["text"]) for r in d)} chữ')
    print(f'  cột rác     : {n_bad} ({100*n_bad/len(d):.1f}%)   [mong đợi < 2%]')
    print(f'  consider=1  : {sum(r["consider"] for r in d)} ({100*sum(r["consider"] for r in d)/len(d):.1f}%)   [~16%]')
    print(f'  giản thể sót: {n_simp}   [phải = 0]')
    print(f'  bản tâm     : {sum(r["middle"] for r in d)}')
    print(f'  volume      : {set(r["volume"] for r in d)}')
    print(f'  schema OK   : {all(tuple(r.keys()) == KEYS for r in d)}')
    print('  mẫu:', ' | '.join(r['text'][:22] for r in d[:3]))

for f in sorted(glob.glob('/kaggle/working/*.json') + glob.glob('output/han_ocr/*.json')):
    report(f)

## 8. Tải kết quả về

`/kaggle/working/` **mất khi session kết thúc**. Nhớ tải file về hoặc Save Version.

In [ ]:
!mkdir -p /kaggle/working/ocr_result
!cp -f /kaggle/working/*.json /kaggle/working/ocr_result/ 2>/dev/null
!cp -f output/han_ocr/*.json /kaggle/working/ocr_result/ 2>/dev/null
!cd /kaggle/working && zip -qr ocr_result.zip ocr_result
!ls -la /kaggle/working/ocr_result.zip /kaggle/working/ocr_result/

---
## Lỗi hay gặp

| Triệu chứng | Xử lý |
|---|---|
| `assert` ở cell 3 fail | Cài nhầm bản CPU — chạy lại cell 2, xem `tag` CUDA có đúng không |
| `assert` ở cell clone fail | Code chưa push. Chạy `git add main.py ocr/ core/ scripts/ requirements.txt && git commit && git push` ở máy local |
| `ModuleNotFoundError: scipy` | `ocr_pipeline.py` import scipy ở module level — chạy lại cell cài đặt |
| PDF không ra gì, exit code 0 | Thiếu `pymupdf` → `main.py` bỏ qua PDF trong im lặng |
| `ModuleNotFoundError: core.interfaces` | Chỉ ảnh hưởng segment/align, không ảnh hưởng OCR. Cần commit `core/` |
| Chạy chậm như CPU | Xem lại cell 3 — gần như chắc chắn Paddle không thấy GPU |
| OOM trên GPU | Chạy 1 shard: bỏ `--shard`, dùng `--device gpu:0` |

## Chưa kiểm chứng

- Notebook này **chưa chạy thật trên Kaggle** (máy local không có GPU NVIDIA).
  Phần đã kiểm: cú pháp, CLI, `--device`, shard/merge, tương thích Python 3.12.
  Phần chỉ xác minh được khi anh chạy: cài `paddlepaddle-gpu` và tốc độ GPU.
- `14_15.pdf` chưa được test (tôi chỉ test `02.pdf`). Chạy 5 trang trước,
  và để ý `bản tâm` có ra không — quy tắc chẵn/lẻ có vẻ không đúng với bộ PDF này.
- Mọi con số chất lượng đều là proxy, **chưa có CER thật**.